In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 10 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 10.C1 — Measure a uniform perturbation

For equally shaped finite arrays f,g return their maximum absolute entry difference.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Take the largest difference, not an average or a Euclidean norm of all pixels.

</details>

In [3]:
def sup_difference(f,g):
    return float(np.max(np.abs(np.asarray(f)-np.asarray(g))))

In [4]:
assert np.isclose(sup_difference([0,.5],[.1,.45]),.1)
image,sample_id=digit_example(0);f=1-image/16
noise=np.random.default_rng(10).uniform(-.03,.03,size=f.shape)
g=np.clip(f+noise,0,1)
assert sup_difference(f,g)<=.03+1e-12
print("Measured uniform image change:",sup_difference(f,g))

Measured uniform image change: 0.02922745631340562


**Why this works and what it does not establish:** The fixed-domain sublevel-set stability bound uses the uniform norm. Replacing it by a mean absolute error can make an invalid bound look much smaller.

## 10.C2 — Distance from a point to the diagonal

For a finite birth<=death pair return its L-infinity distance to the diagonal.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The best diagonal coordinate lies halfway between birth and death.

</details>

In [5]:
def diagonal_cost(birth,death):
    return (death-birth)/2

In [6]:
assert diagonal_cost(1,5)==2
assert diagonal_cost(3,3)==0
print("A lifetime-four feature costs two to match to the diagonal")

A lifetime-four feature costs two to match to the diagonal


**Why this works and what it does not establish:** Matching to the diagonal represents leaving a feature unmatched to any off-diagonal feature. The factor one-half is essential.

## 10.C3 — Solve the one-point bottleneck problem

For two finite diagram points a and b, compare matching them to each other with matching both separately to the diagonal.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

There are two relevant assignment types; the bottleneck cost of an assignment is its worst pair.

</details>

In [7]:
def one_point_bottleneck(a,b):
    a=np.asarray(a,dtype=float);b=np.asarray(b,dtype=float)
    direct=float(np.max(abs(a-b)))
    separate=max((a[1]-a[0])/2,(b[1]-b[0])/2)
    return min(direct,separate)

In [8]:
assert one_point_bottleneck([0,2],[5,7])==1
for a,b in [([0,2],[.1,2.2]),([0,1],[3,5])]:
    assert np.isclose(one_point_bottleneck(a,b),finite_bottleneck([a],[b]))
print("Direct and diagonal alternatives checked")

Direct and diagonal alternatives checked


**Why this works and what it does not establish:** For larger diagrams, greedy nearest-neighbor pairing is not generally sufficient. The core implementation solves a thresholded perfect-matching problem instead.

## 10.C4 — Check fixed-grid stability on real data

For one image and a same-shape finite perturbation of its entrance array, return measured bottleneck H1 distance and uniform input difference.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Keep the pixel domain and triangulation fixed. Complete the filtration before taking its finite H1 diagram.

</details>

In [9]:
def image_stability_pair(image,perturbation):
    f=1-np.asarray(image,dtype=float)/16
    g=np.clip(f+perturbation,0,1)
    D=diagram(persistent_homology(pixel_filtration(f)),1,finite_only=True)
    E=diagram(persistent_homology(pixel_filtration(g)),1,finite_only=True)
    return finite_bottleneck(D,E),float(np.max(abs(f-g)))

In [10]:
image,sample_id=digit_example(8)
noise=np.random.default_rng(104).uniform(-.04,.04,size=image.shape)
bound_distance,uniform=image_stability_pair(image,noise)
assert bound_distance<=uniform+1e-10
print("Training image",sample_id,"diagram distance",bound_distance,"uniform change",uniform)

Training image 8 diagram distance 0.019541755889131873 uniform change 0.03891396892015959


**Why this works and what it does not establish:** This is an empirical check of an applicable mathematical bound, not a statistical significance test. Arbitrary cropping or registration changes are not covered by this exact experiment.

## 10.C5 — Separate a bound from a discovery

Return whether all measured distances satisfy their paired nonnegative bounds within tolerance. This function checks inequalities only.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Compare corresponding entries, not the averages of the two lists.

</details>

In [11]:
def bounds_hold(distances,bounds,tolerance=1e-10):
    return bool(np.all(np.asarray(distances)<=np.asarray(bounds)+tolerance))

In [12]:
assert bounds_hold([.01,.03],[.02,.04])
assert not bounds_hold([.05],[.01])
print("A passed bound is not automatically biological or predictive evidence")

A passed bound is not automatically biological or predictive evidence


**Why this works and what it does not establish:** Even a perfectly stable descriptor can be irrelevant to a task. To claim usefulness, specify a task, comparators, data-generating assumptions, and evaluation that were not chosen after seeing the result.